# Step 2: Momentum Indicator Research

**Goal**: Find which momentum indicator best predicts "first leg wins" for SurefireHedge V2.

**Metric**: P(price moves `tp_pips` in the signal direction before moving `hedge_pips` against it)

**Candidates**:
1. EMA crossover (fast > slow)
2. RSI above/below 50
3. MACD histogram sign
4. ADX > threshold (trend confirmation)
5. Price > SMA (simple trend)

**Data**: 2+ years EUR-USD candles

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Run from project root so qengine can find .env
os.chdir('/Users/naresh/Documents/Research/qengine')
sys.path.insert(0, '.')

import qengine.indicators as ta
import qengine.helpers as jh
from qengine.research import get_candles

plt.rcParams['figure.figsize'] = (14, 6)
sns.set_style('whitegrid')
print('Setup complete.')

## 1. Load Candle Data

Loading EUR-USD candles. Adjust exchange name and dates to match your database.

In [ ]:
# ── CONFIG: adjust these to match your data ──
EXCHANGE  = 'OANDA'
SYMBOL    = 'EUR-USD'
TIMEFRAME = '5m'           # 5m is practical for surefire cycles
START     = '2024-01-01'
END       = '2026-03-01'

start_ts  = jh.date_to_timestamp(START)
finish_ts = jh.date_to_timestamp(END)

# get_candles returns (warmup_candles, trading_candles)
warmup_candles, candles = get_candles(EXCHANGE, SYMBOL, TIMEFRAME, start_ts, finish_ts, warmup_candles_num=210)

# Prepend warmup if available (indicators need lookback)
if warmup_candles is not None and warmup_candles.ndim == 2 and len(warmup_candles) > 0:
    candles = np.concatenate([warmup_candles, candles], axis=0)
    print(f'Prepended {len(warmup_candles)} warmup candles')

print(f'Loaded {len(candles):,} candles')
print(f'Date range: {jh.timestamp_to_date(int(candles[0][0]))} -> {jh.timestamp_to_date(int(candles[-1][0]))}')
print(f'Shape: {candles.shape}')

## 2. Compute Indicators (sequential arrays)

Pre-compute all candidate indicators across the full candle history.

In [ ]:
FAST_EMA   = 8
SLOW_EMA   = 21
RSI_PERIOD = 14
SMA_PERIOD = 20
ADX_PERIOD = 14
ATR_PERIOD = 14

ema_fast   = ta.ema(candles, period=FAST_EMA, sequential=True)
ema_slow   = ta.ema(candles, period=SLOW_EMA, sequential=True)
rsi_arr    = ta.rsi(candles, period=RSI_PERIOD, sequential=True)
macd_res   = ta.macd(candles, fast_period=FAST_EMA, slow_period=SLOW_EMA, signal_period=9, sequential=True)
adx_arr    = ta.adx(candles, period=ADX_PERIOD, sequential=True)
atr_arr    = ta.atr(candles, period=ATR_PERIOD, sequential=True)
sma_arr    = ta.sma(candles, period=SMA_PERIOD, sequential=True)

closes = candles[:, 2]
highs  = candles[:, 3]
lows   = candles[:, 4]

print(f'Indicators computed for {len(candles):,} bars')
print(f'ATR range: {np.nanmin(atr_arr):.5f} - {np.nanmax(atr_arr):.5f}')
print(f'ADX range: {np.nanmin(adx_arr):.1f} - {np.nanmax(adx_arr):.1f}')

## 3. Define the "First Leg Wins" Simulation

For each bar `i`, we simulate: *if we entered at `close[i]` in the signal direction, would price hit TP before hitting the hedge (SL)?*

- **TP distance** = `ATR[i] * tp_atr_multiple`
- **Hedge distance** = `TP / risk_reward`
- We scan forward bar-by-bar using high/low to check which gets hit first

In [ ]:
TP_ATR_MULTIPLE = 0.8
RISK_REWARD     = 2.0
MAX_BARS_AHEAD  = 100
PIP_SIZE        = 0.0001

def simulate_first_leg(entry_price, direction, tp_dist, hedge_dist, future_highs, future_lows):
    if direction == 'long':
        tp_price    = entry_price + tp_dist
        hedge_price = entry_price - hedge_dist
        for j in range(len(future_highs)):
            hit_tp    = future_highs[j] >= tp_price
            hit_hedge = future_lows[j] <= hedge_price
            if hit_tp and hit_hedge:
                return 'loss'
            if hit_tp:
                return 'win'
            if hit_hedge:
                return 'loss'
    else:
        tp_price    = entry_price - tp_dist
        hedge_price = entry_price + hedge_dist
        for j in range(len(future_lows)):
            hit_tp    = future_lows[j] <= tp_price
            hit_hedge = future_highs[j] >= hedge_price
            if hit_tp and hit_hedge:
                return 'loss'
            if hit_tp:
                return 'win'
            if hit_hedge:
                return 'loss'
    return 'timeout'

def run_simulation(signals, closes, highs, lows, atr_vals, tp_mult, rr, max_bars):
    num = len(signals)
    outcomes = np.full(num, '', dtype='U7')
    for i in range(num - max_bars):
        if signals[i] == 'skip':
            outcomes[i] = 'skip'
            continue
        atr_val = atr_vals[i]
        if np.isnan(atr_val) or atr_val <= 0:
            outcomes[i] = 'skip'
            continue
        tp_dist    = atr_val * tp_mult
        hedge_dist = tp_dist / rr
        tp_dist    = max(tp_dist, 3.0 * PIP_SIZE)
        hedge_dist = max(hedge_dist, 1.5 * PIP_SIZE)
        end = min(i + 1 + max_bars, num)
        outcomes[i] = simulate_first_leg(
            closes[i], signals[i], tp_dist, hedge_dist, highs[i+1:end], lows[i+1:end]
        )
    return outcomes

print('Simulation functions defined.')

## 4. Generate Signals for Each Indicator

In [ ]:
N_BARS = len(candles)

def make_signals(long_cond, short_cond):
    sigs = np.full(len(long_cond), 'skip', dtype='U5')
    sigs[long_cond]  = 'long'
    sigs[short_cond] = 'short'
    return sigs

sig_ema = make_signals(ema_fast > ema_slow, ema_fast < ema_slow)
sig_rsi = make_signals(rsi_arr > 50, rsi_arr < 50)
macd_hist = macd_res[2] if isinstance(macd_res, tuple) else macd_res.hist
sig_macd = make_signals(macd_hist > 0, macd_hist < 0)
sig_sma = make_signals(closes > sma_arr, closes < sma_arr)

ADX_THRESHOLD = 20.0
adx_ok = adx_arr >= ADX_THRESHOLD
sig_ema_adx = sig_ema.copy()
sig_ema_adx[~adx_ok] = 'skip'

sig_always_long  = np.full(N_BARS, 'long', dtype='U5')
sig_always_short = np.full(N_BARS, 'short', dtype='U5')

indicators = {
    'EMA Cross':    sig_ema,
    'RSI Midline':  sig_rsi,
    'MACD Hist':    sig_macd,
    'Price > SMA':  sig_sma,
    'EMA + ADX':    sig_ema_adx,
    'Always Long':  sig_always_long,
    'Always Short': sig_always_short,
}

for name, sigs in indicators.items():
    longs  = np.sum(sigs == 'long')
    shorts = np.sum(sigs == 'short')
    skips  = np.sum(sigs == 'skip')
    print(f'{name:18s}  long={longs:>7,}  short={shorts:>7,}  skip={skips:>7,}')

## 5. Run First-Leg Simulation for Each Indicator

In [ ]:
from time import time

results = {}

for name, sigs in indicators.items():
    t0 = time()
    outcomes = run_simulation(sigs, closes, highs, lows, atr_arr, TP_ATR_MULTIPLE, RISK_REWARD, MAX_BARS_AHEAD)
    wins     = np.sum(outcomes == 'win')
    losses   = np.sum(outcomes == 'loss')
    timeouts = np.sum(outcomes == 'timeout')
    total    = wins + losses
    win_rate = wins / total * 100 if total > 0 else 0
    results[name] = {
        'wins': wins, 'losses': losses, 'timeouts': timeouts,
        'total': total, 'win_rate': win_rate, 'outcomes': outcomes,
    }
    elapsed = time() - t0
    print(f'{name:18s}  W={wins:>7,}  L={losses:>7,}  T/O={timeouts:>5,}  Win%={win_rate:5.1f}%  ({elapsed:.1f}s)')

print('\n-- Summary --')
for name, r in sorted(results.items(), key=lambda x: -x[1]['win_rate']):
    print(f'  {name:18s}  Win Rate = {r["win_rate"]:.1f}%  (n={r["total"]:,})')

## 6. Visualize: Win Rate Comparison

In [ ]:
df_results = pd.DataFrame([
    {'Indicator': name, 'Win Rate (%)': r['win_rate'], 'Signals': r['total']}
    for name, r in results.items()
]).sort_values('Win Rate (%)', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#e74c3c' if wr < 50 else '#f39c12' if wr < 65 else '#2ecc71'
          for wr in df_results['Win Rate (%)']]
axes[0].barh(df_results['Indicator'], df_results['Win Rate (%)'], color=colors, edgecolor='white')
axes[0].axvline(x=50, color='red', linestyle='--', alpha=0.5, label='50% (coin flip)')
axes[0].axvline(x=65, color='green', linestyle='--', alpha=0.5, label='65% (target)')
axes[0].set_xlabel('Level 0 Win Rate (%)')
axes[0].set_title('First Leg Win Rate by Indicator')
axes[0].legend()
axes[0].set_xlim(0, 100)

for idx, (wr, sig_ct) in enumerate(zip(df_results['Win Rate (%)'], df_results['Signals'])):
    axes[0].text(wr + 1, idx, f'{wr:.1f}% (n={sig_ct:,})', va='center', fontsize=9)

axes[1].barh(df_results['Indicator'], df_results['Signals'], color='steelblue', edgecolor='white')
axes[1].set_xlabel('Number of Signals (trades simulated)')
axes[1].set_title('Signal Frequency')

plt.tight_layout()
plt.savefig('notebooks/surefire_v2/01_win_rate_comparison.png', dpi=100)
plt.show()
print('Chart saved.')

## 7. Indicator Combinations

Test whether combining two indicators (agreement required) improves win rate.

In [ ]:
def combine_signals(sig_a, sig_b):
    combined = np.full(len(sig_a), 'skip', dtype='U5')
    agree_long  = (sig_a == 'long') & (sig_b == 'long')
    agree_short = (sig_a == 'short') & (sig_b == 'short')
    combined[agree_long]  = 'long'
    combined[agree_short] = 'short'
    return combined

combos = {
    'EMA + RSI':        combine_signals(sig_ema, sig_rsi),
    'EMA + MACD':       combine_signals(sig_ema, sig_macd),
    'EMA + SMA':        combine_signals(sig_ema, sig_sma),
    'RSI + MACD':       combine_signals(sig_rsi, sig_macd),
    'EMA + RSI + ADX':  combine_signals(combine_signals(sig_ema, sig_rsi), sig_ema_adx),
    'EMA + MACD + ADX': combine_signals(combine_signals(sig_ema, sig_macd), sig_ema_adx),
}

combo_results = {}
for name, sigs in combos.items():
    outcomes = run_simulation(sigs, closes, highs, lows, atr_arr, TP_ATR_MULTIPLE, RISK_REWARD, MAX_BARS_AHEAD)
    wins   = np.sum(outcomes == 'win')
    losses = np.sum(outcomes == 'loss')
    total  = wins + losses
    wr     = wins / total * 100 if total > 0 else 0
    combo_results[name] = {'wins': wins, 'losses': losses, 'total': total, 'win_rate': wr}
    print(f'{name:22s}  Win%={wr:5.1f}%  (n={total:,})')

print('\n-- Best combinations --')
for name, r in sorted(combo_results.items(), key=lambda x: -x[1]['win_rate']):
    print(f'  {name:22s}  Win Rate = {r["win_rate"]:.1f}%  (n={r["total"]:,})')

## 8. Parameter Sensitivity: EMA Period Sweep

In [ ]:
fast_periods = [5, 8, 10, 13, 15, 20]
slow_periods = [15, 21, 30, 50, 75, 100]

ema_sweep = []
for fp in fast_periods:
    for sp in slow_periods:
        if fp >= sp:
            continue
        ef = ta.ema(candles, period=fp, sequential=True)
        es = ta.ema(candles, period=sp, sequential=True)
        sigs = make_signals(ef > es, ef < es)
        outcomes = run_simulation(sigs, closes, highs, lows, atr_arr, TP_ATR_MULTIPLE, RISK_REWARD, MAX_BARS_AHEAD)
        wins   = np.sum(outcomes == 'win')
        losses = np.sum(outcomes == 'loss')
        total  = wins + losses
        wr     = wins / total * 100 if total > 0 else 0
        ema_sweep.append({'fast': fp, 'slow': sp, 'win_rate': wr, 'signals': total})

df_ema = pd.DataFrame(ema_sweep).sort_values('win_rate', ascending=False)
print(df_ema.to_string(index=False))

pivot = df_ema.pivot(index='fast', columns='slow', values='win_rate')
plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=50, vmin=25, vmax=45)
plt.title('EMA Cross Win Rate (%) - Fast vs Slow Period')
plt.xlabel('Slow EMA Period')
plt.ylabel('Fast EMA Period')
plt.savefig('notebooks/surefire_v2/01_ema_heatmap.png', dpi=100)
plt.show()

## 9. ADX Threshold Sweep

How does the ADX threshold affect the tradeoff between signal quality and quantity?

In [ ]:
adx_thresholds = [10, 15, 20, 25, 30, 35, 40]
adx_sweep = []

for thresh in adx_thresholds:
    sigs = sig_ema.copy()
    sigs[adx_arr < thresh] = 'skip'
    outcomes = run_simulation(sigs, closes, highs, lows, atr_arr, TP_ATR_MULTIPLE, RISK_REWARD, MAX_BARS_AHEAD)
    wins   = np.sum(outcomes == 'win')
    losses = np.sum(outcomes == 'loss')
    total  = wins + losses
    wr     = wins / total * 100 if total > 0 else 0
    adx_sweep.append({'adx_threshold': thresh, 'win_rate': wr, 'signals': total})

df_adx = pd.DataFrame(adx_sweep)
print(df_adx.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.plot(df_adx['adx_threshold'], df_adx['win_rate'], 'g-o', linewidth=2, label='Win Rate (%)')
ax2.bar(df_adx['adx_threshold'], df_adx['signals'], alpha=0.3, color='steelblue', width=2, label='Signal Count')
ax1.set_xlabel('ADX Threshold')
ax1.set_ylabel('Win Rate (%)', color='green')
ax2.set_ylabel('Number of Signals', color='steelblue')
ax1.axhline(y=65, color='green', linestyle='--', alpha=0.4, label='65% target')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.title('ADX Threshold: Win Rate vs Signal Count')
plt.savefig('notebooks/surefire_v2/01_adx_sweep.png', dpi=100)
plt.show()

## 10. Win Rate Over Time (Stability Check)

A good indicator should have stable win rate across different market periods.

In [ ]:
timestamps = candles[:, 0]

def rolling_win_rate(outcomes, timestamps, window_bars=5000):
    wins = (outcomes == 'win').astype(float)
    losses = (outcomes == 'loss').astype(float)
    resolved = wins + losses
    cum_wins = np.cumsum(wins)
    cum_resolved = np.cumsum(resolved)
    roll_wins = cum_wins[window_bars:] - cum_wins[:-window_bars]
    roll_total = cum_resolved[window_bars:] - cum_resolved[:-window_bars]
    with np.errstate(divide='ignore', invalid='ignore'):
        roll_wr = np.where(roll_total > 0, roll_wins / roll_total * 100, np.nan)
    return timestamps[window_bars:], roll_wr

top_indicators = ['EMA Cross', 'EMA + ADX', 'RSI Midline', 'MACD Hist', 'Price > SMA']

fig, ax = plt.subplots(figsize=(16, 6))
for name in top_indicators:
    ts, wr = rolling_win_rate(results[name]['outcomes'], timestamps, window_bars=5000)
    dates = pd.to_datetime(ts, unit='ms')
    ax.plot(dates, wr, label=name, alpha=0.8)

ax.axhline(y=65, color='green', linestyle='--', alpha=0.4, label='65% target')
ax.axhline(y=50, color='red', linestyle='--', alpha=0.3, label='50% baseline')
ax.set_xlabel('Date')
ax.set_ylabel('Rolling Win Rate (%)')
ax.set_title(f'Rolling Win Rate (5000-bar window) - TP mult={TP_ATR_MULTIPLE}, RR={RISK_REWARD}')
ax.legend(loc='lower left')
ax.set_ylim(20, 80)
plt.tight_layout()
plt.savefig('notebooks/surefire_v2/01_rolling_win_rate.png', dpi=100)
plt.show()

## 11. Combined Scoreboard

Final ranking considering win rate, signal count, and stability.

In [ ]:
all_results = {**results, **combo_results}

scoreboard = []
for name, r in all_results.items():
    outcomes = r.get('outcomes')
    if outcomes is not None:
        _, roll_wr = rolling_win_rate(outcomes, timestamps, window_bars=5000)
        stability = np.nanstd(roll_wr)
    else:
        stability = np.nan
    scoreboard.append({
        'Indicator': name,
        'Win Rate (%)': round(r['win_rate'], 1),
        'Signals': r['total'],
        'Stability (std)': round(stability, 1) if not np.isnan(stability) else '-',
    })

df_score = pd.DataFrame(scoreboard).sort_values('Win Rate (%)', ascending=False)
print(df_score.to_string(index=False))

print('\n-- Key Observation --')
print(f'With TP_ATR_MULTIPLE={TP_ATR_MULTIPLE} and RISK_REWARD={RISK_REWARD}:')
print(f'  TP distance = {RISK_REWARD}x hedge distance')
print(f'  Expected random-walk win rate = 1/{RISK_REWARD+1:.0f} = {100/(RISK_REWARD+1):.0f}%')
print(f'  Observed baseline (Always Long/Short) ~ {results["Always Long"]["win_rate"]:.1f}%')
print()
print('If all indicators cluster near the baseline, the TP/hedge ratio')
print('dominates over indicator choice. Step 3 (ATR distance sweep) is critical.')